# 02. Long Short-Term Memory (Hochreiter & Schmidhuber, 1997)

Interactive walkthrough accompanying `02_lstm_cell_from_scratch.py`.

We cover:
1. Why Elman RNNs lose gradients exponentially over long horizons.
2. How splitting memory into cell state ($c_t$) and hidden state ($h_t$) creates the **Constant Error Carousel (CEC)**.
3. Exact numerical verification against `torch.nn.LSTMCell`.
4. Plotting Vanilla RNN vs. LSTM gradient flow across $T = 100$ steps.
5. Solving the 1997 Adding Problem and plotting the learned input/forget gates over time.

## 1. Setup

Import PyTorch, NumPy, and Matplotlib.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## 2. LSTM Cell Equations & Intuition

At timestep $t$, given input $\mathbf{x}_t \in \mathbb{R}^{B \times d_x}$ and previous states $(\mathbf{h}_{t-1}, \mathbf{c}_{t-1})$:

$$\begin{bmatrix}\tilde{\mathbf{i}}_t \\ \tilde{\mathbf{f}}_t \\ \tilde{\mathbf{g}}_t \\ \tilde{\mathbf{o}}_t\end{bmatrix} = \mathbf{x}_t \mathbf{W}_{ih} + \mathbf{h}_{t-1}\mathbf{W}_{hh} + \mathbf{b}$$

$$\mathbf{i}_t = \sigma(\tilde{\mathbf{i}}_t), \quad \mathbf{f}_t = \sigma(\tilde{\mathbf{f}}_t), \quad \mathbf{g}_t = \tanh(\tilde{\mathbf{g}}_t), \quad \mathbf{o}_t = \sigma(\tilde{\mathbf{o}}_t)$$

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t \qquad \text{(Constant Error Carousel)}$$

$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

* **Forget gate ($f_t$):** Controls how much of $\mathbf{c}_{t-1}$ survives to step $t$. Initializing the forget bias positively ($b_f = 3.0 \implies f_t \approx 0.95$) starts the memory highway open.
* **Input gate ($i_t$):** Decides when to write new candidate $\mathbf{g}_t$ into $\mathbf{c}_t$ and when to block distractor noise ($i_t \approx 0$).
* **Output gate ($o_t$):** Controls how much of $\tanh(\mathbf{c}_t)$ is exposed to $\mathbf{h}_t$.

In [ ]:
class LSTMCellFromScratch(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, forget_bias_init: float = 3.0):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.W_ih = nn.Parameter(torch.empty(input_dim, 4 * hidden_dim))
        self.W_hh = nn.Parameter(torch.empty(hidden_dim, 4 * hidden_dim))
        self.b = nn.Parameter(torch.zeros(4 * hidden_dim))
        self.reset_parameters(forget_bias_init)

    def reset_parameters(self, forget_bias_init: float):
        bound = 1.0 / (self.hidden_dim ** 0.5)
        nn.init.uniform_(self.W_ih, -bound, bound)
        nn.init.uniform_(self.W_hh, -bound, bound)
        nn.init.uniform_(self.b, -bound, bound)
        with torch.no_grad():
            self.b[self.hidden_dim : 2 * self.hidden_dim].fill_(forget_bias_init)

    def forward(self, x_t: torch.Tensor, state=None, return_gates: bool = False):
        B = x_t.shape[0]
        if state is None:
            h_prev = torch.zeros(B, self.hidden_dim, device=x_t.device, dtype=x_t.dtype)
            c_prev = torch.zeros(B, self.hidden_dim, device=x_t.device, dtype=x_t.dtype)
        else:
            h_prev, c_prev = state

        gates = x_t @ self.W_ih + h_prev @ self.W_hh + self.b
        i_raw, f_raw, g_raw, o_raw = gates.chunk(4, dim=-1)

        i_t = torch.sigmoid(i_raw)
        f_t = torch.sigmoid(f_raw)
        g_t = torch.tanh(g_raw)
        o_t = torch.sigmoid(o_raw)

        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        if return_gates:
            return h_t, c_t, (i_t, f_t, o_t)
        return h_t, c_t

print("LSTMCellFromScratch ready.")

## 3. Equivalence Verification vs. `torch.nn.LSTMCell`

Copy the exact weights of `LSTMCellFromScratch` into PyTorch's built-in `nn.LSTMCell` and check that both forward states (`h_t`, `c_t`) and backward gradients agree within floating-point precision (`< 1e-6`).

In [ ]:
B, T, D, H = 4, 6, 5, 8
x = torch.randn(B, T, D)

custom = LSTMCellFromScratch(D, H)
ref = nn.LSTMCell(D, H)

with torch.no_grad():
    ref.weight_ih.copy_(custom.W_ih.T)
    ref.weight_hh.copy_(custom.W_hh.T)
    ref.bias_ih.copy_(custom.b)
    ref.bias_hh.zero_()

h_c, c_c = torch.zeros(B, H), torch.zeros(B, H)
h_r, c_r = torch.zeros(B, H), torch.zeros(B, H)
for t in range(T):
    h_c, c_c = custom(x[:, t, :], (h_c, c_c))
    h_r, c_r = ref(x[:, t, :], (h_r, c_r))

h_diff = (h_c - h_r).abs().max().item()
c_diff = (c_c - c_r).abs().max().item()
h_c.pow(2).sum().backward()
h_r.pow(2).sum().backward()
grad_diff = (custom.W_hh.grad - ref.weight_hh.grad.T).abs().max().item()

print(f"h_diff={h_diff:.2e} | c_diff={c_diff:.2e} | grad_diff={grad_diff:.2e}")
assert h_diff < 1e-6 and c_diff < 1e-6 and grad_diff < 1e-5

## 4. Gradient Highway Plot: Vanilla RNN vs. LSTM CEC ($T = 80$)

Evaluate a scalar loss **only at the final timestep** $t = T - 1$ and plot the gradient norm reaching every earlier timestep $t = 0 \dots T - 1$ on a log scale.  
In a Vanilla RNN, gradients vanish exponentially toward floating-point zero. Along the LSTM cell state $c_t$, the gradient remains nearly constant.

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W_ih = nn.Linear(input_dim, hidden_dim, bias=False)
        self.W_hh = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.head = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        B, T, _ = x.shape
        h_t = torch.zeros(B, self.hidden_dim)
        states = []
        for t in range(T):
            h_t = torch.tanh(self.W_ih(x[:, t, :]) + self.W_hh(h_t))
            if h_t.requires_grad:
                h_t.retain_grad()
            states.append(h_t)
        return self.head(h_t), states

class LSTMModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.cell = LSTMCellFromScratch(input_dim, hidden_dim, forget_bias_init=3.0)
        self.head = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, return_gates=False):
        B, T, _ = x.shape
        h_t = torch.zeros(B, self.cell.hidden_dim)
        c_t = torch.zeros(B, self.cell.hidden_dim)
        c_states, gate_log = [], []
        for t in range(T):
            h_t, c_t, (i_t, f_t, o_t) = self.cell(x[:, t, :], (h_t, c_t), return_gates=True)
            if c_t.requires_grad:
                c_t.retain_grad()
            c_states.append(c_t)
            if return_gates:
                gate_log.append((i_t.mean().item(), f_t.mean().item()))
        logits = self.head(h_t)
        if return_gates:
            return logits, c_states, gate_log
        return logits, c_states

T_test = 80
x_test = torch.randn(4, T_test, 8)

rnn = VanillaRNN(8, 16, 2)
rnn_out, rnn_states = rnn(x_test)
rnn_out.sum().backward()
rnn_norms = [s.grad.norm().item() + 1e-16 for s in rnn_states]

lstm = LSTMModel(8, 16, 2)
lstm_out, lstm_c_states = lstm(x_test)
lstm_out.sum().backward()
lstm_norms = [s.grad.norm().item() + 1e-16 for s in lstm_c_states]

plt.figure(figsize=(9, 4))
plt.semilogy(rnn_norms, label="Vanilla RNN ||dL / dh_t||", color="#e74c3c", lw=2)
plt.semilogy(lstm_norms, label="LSTM CEC ||dL / dc_t||", color="#2980b9", lw=2)
plt.title("Gradient Flow Backward From Final Step (T = 80)")
plt.xlabel("Timestep t (0 = start, 79 = loss step)")
plt.ylabel("Gradient Norm (log scale)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Solving the 1997 Adding Problem & Inspecting Learned Gates

Each sample has $T = 50$ steps of $(v_t, m_t)$ where $v_t \sim \text{Uniform}(0, 1)$ and $m_t = 1$ at exactly two positions (all other 48 steps are distractor noise with $m_t = 0$). Target: $v_{p_1} + v_{p_2}$.

* Constant mean guess ($1.0$) gives $\text{MSE} = \frac{1}{12} + \frac{1}{12} = 0.1667$.
* Watch both models train, then inspect the LSTM's average **input gate** $i_t$ and **forget gate** $f_t$ on one validation example to see gating in action.

In [ ]:
seq_len = 50
batch_size = 64

def sample_adding_batch(n=batch_size):
    vals = torch.rand(n, seq_len, 1)
    mask = torch.zeros(n, seq_len, 1)
    p1 = torch.randint(0, seq_len // 2, (n,))
    p2 = torch.randint(seq_len // 2, seq_len, (n,))
    idx = torch.arange(n)
    mask[idx, p1, 0] = 1.0
    mask[idx, p2, 0] = 1.0
    return torch.cat([vals, mask], dim=-1), (vals * mask).sum(dim=1)

def train_adding(model, steps=700, lr=5e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = []
    for _ in range(steps):
        xb, yb = sample_adding_batch()
        opt.zero_grad()
        out, _ = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        history.append(loss.item())
    return history

rnn_add = VanillaRNN(2, 32, 1)
lstm_add = LSTMModel(2, 32, 1)

hist_rnn = train_adding(rnn_add)
hist_lstm = train_adding(lstm_add)

with torch.no_grad():
    xv, yv = sample_adding_batch(n=512)
    mse_rnn = nn.functional.mse_loss(rnn_add(xv)[0], yv).item()
    mse_lstm = nn.functional.mse_loss(lstm_add(xv)[0], yv).item()

print(f"Final Val MSE | VanillaRNN={mse_rnn:.5f} | LSTM={mse_lstm:.5f} (baseline=0.1667)")

plt.figure(figsize=(9, 4))
plt.plot(hist_rnn, label=f"VanillaRNN (val MSE={mse_rnn:.4f})", alpha=0.8)
plt.plot(hist_lstm, label=f"LSTM (val MSE={mse_lstm:.4f})", alpha=0.9)
plt.axhline(1.0 / 6.0, color="gray", linestyle="--", label="Constant-mean baseline (0.1667)")
plt.title("1997 Adding Problem Training Curves (T = 50)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Gate Activity Inspection on a Single Sequence

Finally, feed **one** sequence into the trained LSTM and plot mean input gate $i_t$ and forget gate $f_t$ alongside the two target marker spikes ($m_t = 1$). Notice how $i_t$ spikes specifically when the marker turns on while $f_t$ stays high to preserve the running sum.

In [ ]:
x_one, y_one = sample_adding_batch(n=1)
with torch.no_grad():
    pred, _, gate_log = lstm_add(x_one, return_gates=True)

i_vals = [g[0] for g in gate_log]
f_vals = [g[1] for g in gate_log]
markers = x_one[0, :, 1].numpy()

print(f"Target sum={y_one.item():.4f} | LSTM prediction={pred.item():.4f}")

plt.figure(figsize=(9, 4))
plt.plot(i_vals, label="Mean Input Gate (i_t)", lw=2)
plt.plot(f_vals, label="Mean Forget Gate (f_t)", lw=2)
plt.plot(markers, label="Marker Input (m_t)", linestyle="--", color="black", alpha=0.7)
plt.title("Trained LSTM Gate Activations Across Timesteps")
plt.xlabel("Timestep t")
plt.ylabel("Gate / Marker Value")
plt.ylim(-0.05, 1.1)
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()